# 02 · Time Splits and Feature Experiments
| Stage | Fit on | Validate or evaluate on |
|---|---|---|
| Fold 1 | 2017 | 2018 |
| Fold 2 | 2017-2018 | 2019 |
| Fold 3 | 2017-2019 | 2020 |
| Final pipeline, after model selection | 2017-2020 | 2021 |

The prepared observations are loaded from `data/processed/`. Feature lists, filtering records and fold summaries are saved under `output/02_time_split_and_preprocessing/`. Use the same configuration in notebooks 01-03.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src/config.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/config.py").exists():
    raise FileNotFoundError("Open this notebook from the project root or notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src.config import ProjectConfig
from src.io_utils import prepare_run, read_json, write_json, ensure_unlocked, record_input_checksums
from src.data import load_development_data, make_feature_sets, make_preprocessor, chronological_folds

cfg = ProjectConfig()
RUN = prepare_run(ROOT, cfg)
ensure_unlocked(RUN)
record_input_checksums(ROOT, cfg, RUN)
OUT = ROOT / "output" / "02_time_split_and_preprocessing"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Experiment: {OUT.name}\nNotebook results: {OUT}")

Experiment: 02_time_split_and_preprocessing
Notebook results: C:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_2_UnsupervisedLearning\output\02_time_split_and_preprocessing


In [2]:
development_rows, data_audit = load_development_data(ROOT, cfg)
write_json(RUN / "data_audit.json", data_audit)
allowed_features = data_audit["allowed_features"]
selection_rows = development_rows.loc[
    development_rows["Date"].le(pd.Timestamp(cfg.feature_selection_end))
]
feature_sets, feature_catalog, removed_correlations = make_feature_sets(
    selection_rows=selection_rows, allowed_features=allowed_features, config=cfg
)
write_json(OUT / "feature_sets.json", feature_sets)
feature_catalog.to_csv(OUT / "feature_catalog.csv")
removed_correlations.to_csv(OUT / "correlated_features_removed.csv", index=False)
write_json(OUT / "feature_filter_fit.json", {
    "start": selection_rows["Date"].min(), "end": selection_rows["Date"].max(),
    "n_rows": len(selection_rows), "selection_end": cfg.feature_selection_end,
})
display(feature_catalog)
display(pd.DataFrame({
    "feature_set": feature_sets.keys(),
    "n_features": [len(features) for features in feature_sets.values()],
}))

,family,missing_fraction,unique,usable
feature,,,,
OpenToCloseReturn,returns,0.000000,96556,True
HighLowRange,risk,0.000000,86215,True
GapReturn,returns,0.000000,73163,True
LogReturn1Day,returns,0.000000,104989,True
Momentum5Days,momentum,0.000000,332477,True
Momentum20Days,momentum,0.000000,339783,True
Momentum60Days,momentum,0.000000,342717,True
Volatility20Days,risk,0.000000,348105,True
Volatility60Days,risk,0.000000,348016,True


,feature_set,n_features
0,A,42
1,B,35
2,C,16
3,D,8


## Walk-forward periods

In [3]:
fold_records = []
for fold_name, training_rows, validation_rows in chronological_folds(development_rows, cfg):
    fold_records.append({
        "fold": fold_name,
        "training_start": training_rows["Date"].min(),
        "training_end": training_rows["Date"].max(),
        "validation_start": validation_rows["Date"].min(),
        "validation_end": validation_rows["Date"].max(),
        "n_training_rows": len(training_rows), "n_validation_rows": len(validation_rows),
    })
fold_summary = pd.DataFrame(fold_records)
fold_summary.to_csv(OUT / "folds.csv", index=False)
display(fold_summary)
print(f"Final refit after selection: {len(development_rows):,} rows through {cfg.development_end}")

,fold,training_start,training_end,validation_start,validation_end,n_training_rows,n_validation_rows
0,validate_2018,2017-03-31,2017-12-29,2018-01-04,2018-12-28,349699,464576
1,validate_2019,2017-03-31,2018-12-28,2019-01-04,2019-12-30,814275,465067
2,validate_2020,2017-03-31,2019-12-30,2020-01-06,2020-12-30,1279342,475518


Final refit after selection: 1,754,860 rows through 2020-12-31


## Preprocessing during tuning

For each feature set and chronological fold, create a fresh preprocessor:

```python
for fold_name, training_rows, validation_rows in chronological_folds(development_rows, cfg):
    preprocessor = make_preprocessor()
    X_train = preprocessor.fit_transform(training_rows[selected_features])
    X_valid = preprocessor.transform(validation_rows[selected_features])
    # Fit GMM candidates on X_train and evaluate them on X_valid.
```